In [ ]:
# =========================================================
# 第二步：
# 1) 读取4个点位库
# 2) 按 Generation Z 权重随机选点
# 3) 将所有选中点两两组合，计算城市步行最短路径
# 4) 画出“所有可能线路”的网络图
# 5) 输出来源库统计
# =========================================================

import os
import math
import time
import random
import itertools
import numpy as np
import pandas as pd
import folium
import osmnx as ox
import networkx as nx

from shapely.geometry import box
from collections import Counter, defaultdict
from IPython.display import display
from folium import FeatureGroup, LayerControl
from folium.plugins import Fullscreen

# =========================================================
# 0. 动态随机种子
# =========================================================
seed = int(time.time() * 1000) % (2**32 - 1)
random.seed(seed)
np.random.seed(seed)
print(f"本次动态随机种子: {seed}")

# =========================================================
# 1. Generation Z 媒介权重
#    每抽1个点时，都按这个概率重新抽来源库
# =========================================================
MEDIA_WEIGHTS = {
    "short video social media": 0.03,
    "website blogs": 0.10,
    "tv official news": 0.22,
    "paper manual magazine": 0.65
}

# =========================================================
# 2. 每类点位需求
#    这里总共是14个点
# =========================================================
CATEGORY_NEEDS = {
    "attraction": 8,
    "restaurant": 2,
    "hotel": 1,
    "shop": 3
}

# =========================================================
# 3. count 权重
#    6 / 4 / 1 只是概率高低，不是绝对优先
# =========================================================
COUNT_WEIGHT_MAP = {
    6: 6.0,
    4: 4.0,
    1: 1.0
}

# =========================================================
# 4. 类别颜色
# =========================================================
CATEGORY_COLORS = {
    "attraction": "#6ec1ff",
    "restaurant": "#ff6b6b",
    "hotel": "#ffffff",
    "shop": "#ffd166"
}

# =========================================================
# 5. 工具函数
# =========================================================
def normalize_media_name(filename):
    name = os.path.splitext(os.path.basename(filename))[0].lower().strip()

    if "short" in name and "video" in name:
        return "short video social media"
    elif "website" in name and "blog" in name:
        return "website blogs"
    elif "tv" in name or "official news" in name:
        return "tv official news"
    elif "paper" in name or "manual" in name or "magazine" in name:
        return "paper manual magazine"
    else:
        raise ValueError(
            f"无法从文件名识别媒介类型：{filename}\n"
            f"请确保文件名中包含以下关键词之一：\n"
            f"short video social media / website blogs / TV official news / Paper manual magazine"
        )

def first_sheet_name(xlsx_path):
    xls = pd.ExcelFile(xlsx_path)
    return xls.sheet_names[0]

def parse_lat_lon(text):
    if pd.isna(text):
        return None, None

    s = str(text).strip().replace("，", ",")
    parts = [p.strip() for p in s.split(",")]

    if len(parts) != 2:
        return None, None

    try:
        lat = float(parts[0])
        lon = float(parts[1])
        return lat, lon
    except:
        return None, None

def count_to_weight(x):
    try:
        x = int(x)
    except:
        return 1.0
    return COUNT_WEIGHT_MAP.get(x, float(max(x, 1)))

def weighted_pick_one(df_sub, weight_col="sample_weight"):
    if len(df_sub) == 0:
        return None

    w = df_sub[weight_col].astype(float).values
    if np.all(w <= 0):
        w = np.ones(len(df_sub), dtype=float)

    p = w / w.sum()
    idx = np.random.choice(df_sub.index, size=1, replace=False, p=p)[0]
    return df_sub.loc[idx]

# =========================================================
# 6. 读取 Excel 并转成长表
#    兼容你当前这种横向四类点位结构
# =========================================================
def excel_to_long_table(path):
    media_name = normalize_media_name(path)
    sheet = first_sheet_name(path)
    df = pd.read_excel(path, sheet_name=sheet)

    restaurant_coord_col = "LA LN.1"
    if "LA LN.1" not in df.columns and "Unnamed: 5" in df.columns:
        restaurant_coord_col = "Unnamed: 5"
    elif "LA LN.1" in df.columns and "Unnamed: 5" in df.columns:
        n1 = df["LA LN.1"].notna().sum()
        n2 = df["Unnamed: 5"].notna().sum()
        restaurant_coord_col = "Unnamed: 5" if n2 > n1 else "LA LN.1"

    category_spec = {
        "attraction": ("attraction", "count", "LA LN"),
        "restaurant": ("restaurant", "count.1", restaurant_coord_col),
        "hotel": ("hotel", "count.2", "LA LN.1" if restaurant_coord_col == "Unnamed: 5" else "LA LN.2"),
        "shop": ("shop", "count.3", "LA LN.2" if restaurant_coord_col == "Unnamed: 5" else "LA LN.3")
    }

    if "LA LN.3" in df.columns:
        category_spec["hotel"] = ("hotel", "count.2", "LA LN.2")
        category_spec["shop"]  = ("shop", "count.3", "LA LN.3")

    rows = []

    for cat, (name_col, count_col, coord_col) in category_spec.items():
        if name_col not in df.columns or count_col not in df.columns or coord_col not in df.columns:
            print(f"警告：{os.path.basename(path)} 缺少列 -> {cat}: {name_col}, {count_col}, {coord_col}")
            continue

        tmp = df[[name_col, count_col, coord_col]].copy()
        tmp.columns = ["name", "count", "coord"]
        tmp = tmp.dropna(subset=["name", "coord"]).copy()

        tmp["name"] = tmp["name"].astype(str).str.strip()
        tmp["count"] = pd.to_numeric(tmp["count"], errors="coerce").fillna(1)

        tmp["lat"] = tmp["coord"].apply(lambda x: parse_lat_lon(x)[0])
        tmp["lon"] = tmp["coord"].apply(lambda x: parse_lat_lon(x)[1])

        tmp = tmp.dropna(subset=["lat", "lon"]).copy()

        tmp["category"] = cat
        tmp["media"] = media_name
        tmp["source_file"] = os.path.basename(path)

        rows.append(tmp[["media", "category", "name", "count", "lat", "lon", "source_file"]])

    if len(rows) == 0:
        return pd.DataFrame(columns=["media", "category", "name", "count", "lat", "lon", "source_file"])

    long_df = pd.concat(rows, ignore_index=True)

    # 同一媒介、同一类别、同名同坐标，保留 count 最大的一条
    long_df["coord_key"] = long_df["lat"].round(8).astype(str) + "," + long_df["lon"].round(8).astype(str)
    long_df = (
        long_df.sort_values(
            ["media", "category", "name", "coord_key", "count"],
            ascending=[True, True, True, True, False]
        )
        .drop_duplicates(subset=["media", "category", "name", "coord_key"], keep="first")
        .drop(columns=["coord_key"])
        .reset_index(drop=True)
    )

    return long_df

# =========================================================
# 7. 汇总四个点位库
# =========================================================
all_tables = []
for f in uploaded_files:
    t = excel_to_long_table(f)
    all_tables.append(t)

points_df = pd.concat(all_tables, ignore_index=True)

if len(points_df) == 0:
    raise ValueError("四个文件读取后没有可用点位，请检查列名和经纬度格式。")

points_df["sample_weight"] = points_df["count"].apply(count_to_weight)

print("\n统一后的点位数据预览：")
display(points_df.head())

print("\n各媒介 / 各类别数量：")
display(points_df.groupby(["media", "category"]).size().unstack(fill_value=0))

# =========================================================
# 8. 选点逻辑
#    每抽一个点：
#    1) 在有库存的媒介库中按 70/18/4/8 抽一个来源库
#    2) 再在这个来源库 + 当前类别中，按 count 权重抽一个点
# =========================================================
def pick_points_for_category(df_all, category, n_need, media_weights):
    df_cat = df_all[df_all["category"] == category].copy()

    if len(df_cat) < n_need:
        raise ValueError(f"{category} 可选点不足，需要 {n_need} 个，但只有 {len(df_cat)} 个。")

    selected_rows = []
    used_idx = set()

    for _ in range(n_need):
        remaining = df_cat.loc[~df_cat.index.isin(used_idx)].copy()

        if len(remaining) == 0:
            raise ValueError(f"{category} 剩余点不足。")

        medias_available = remaining["media"].unique().tolist()

        weights = np.array([media_weights.get(m, 0.0) for m in medias_available], dtype=float)

        if weights.sum() <= 0:
            weights = np.ones(len(medias_available), dtype=float)

        probs = weights / weights.sum()
        chosen_media = np.random.choice(medias_available, p=probs)

        pool = remaining[remaining["media"] == chosen_media].copy()

        # 极端情况下回退
        if len(pool) == 0:
            pool = remaining.copy()

        picked = weighted_pick_one(pool, "sample_weight")
        used_idx.add(picked.name)
        selected_rows.append(picked)

    return pd.DataFrame(selected_rows).reset_index(drop=True)

selected_parts = []
for cat, n in CATEGORY_NEEDS.items():
    sel = pick_points_for_category(points_df, cat, n, MEDIA_WEIGHTS)
    selected_parts.append(sel)

selected_df = pd.concat(selected_parts, ignore_index=True).reset_index(drop=True)

print("\n本次选中的点：")
display(selected_df[["media", "category", "name", "count", "lat", "lon"]])

# =========================================================
# 9. 输出来源库统计
#    每个来源库、每个种类选了多少个点
# =========================================================
source_summary = (
    selected_df.groupby(["media", "category"])
    .size()
    .reset_index(name="selected_count")
    .sort_values(["media", "category"])
    .reset_index(drop=True)
)

print("\n每个来源库、每个种类选了多少个点：")
display(source_summary)

print("\n按来源库透视统计：")
display(
    source_summary.pivot(index="media", columns="category", values="selected_count").fillna(0).astype(int)
)

# =========================================================
# 10. 构建研究范围并下载真实步行路网
# =========================================================
all_pts = list(zip(selected_df["lat"], selected_df["lon"]))
lats = [p[0] for p in all_pts]
lons = [p[1] for p in all_pts]

north = max(lats) + 0.01
south = min(lats) - 0.01
east  = max(lons) + 0.01
west  = min(lons) - 0.01

print("\n正在下载城市步行路网...")

study_area = box(west, south, east, north)

G = ox.graph_from_polygon(
    study_area,
    network_type="walk",
    simplify=True
)

try:
    G = ox.distance.add_edge_lengths(G)
except:
    pass

print("路网下载完成。")
print(f"节点数: {len(G.nodes)}")
print(f"边数: {len(G.edges)}")

# =========================================================
# 11. 将选中点匹配到最近路网节点
# =========================================================
selected_df = selected_df.copy()
selected_df["nearest_node"] = selected_df.apply(
    lambda r: ox.distance.nearest_nodes(G, X=r["lon"], Y=r["lat"]),
    axis=1
)

# =========================================================
# 12. 所有选中点两两组合，计算步行最短路径
#    14个点 -> 91条点对
# =========================================================
all_pairs = list(itertools.combinations(selected_df.index.tolist(), 2))
print(f"\n点对组合数量: {len(all_pairs)}")

all_route_segments = []
edge_frequency = Counter()

for i, j in all_pairs:
    row_i = selected_df.loc[i]
    row_j = selected_df.loc[j]

    node_i = row_i["nearest_node"]
    node_j = row_j["nearest_node"]

    try:
        path_nodes = nx.shortest_path(G, node_i, node_j, weight="length")
        all_route_segments.append(path_nodes)

        # 统计路段被多少条最短路径共用
        for a, b in zip(path_nodes[:-1], path_nodes[1:]):
            edge_key = tuple(sorted((a, b)))
            edge_frequency[edge_key] += 1

    except nx.NetworkXNoPath:
        print(f"警告：{row_i['name']} <-> {row_j['name']} 没有找到步行路径。")

print(f"成功求得最短路径段数: {len(all_route_segments)}")
print(f"被使用的唯一街道路段数: {len(edge_frequency)}")

# =========================================================
# =========================================================
# 13. 画在线地图（只改这一部分）
# 1) 线宽改为原来的 0.5 倍
# 2) 不显示地图上的文字标注
# 3) 地图后输出点位表格：名称 / 来源库 / 类型
# =========================================================
center_lat = np.mean(lats)
center_lon = np.mean(lons)

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles="CartoDB dark_matter",
    control_scale=True
)

Fullscreen().add_to(m)

# 先画所有最短路径网络
# =========================================================
# =========================================================
# 画所有最短路径网络（线宽×2，不透明）
# =========================================================

if len(edge_frequency) > 0:
    max_freq = max(edge_frequency.values())
else:
    max_freq = 1

for (u, v), freq in edge_frequency.items():
    coords = [
        (G.nodes[u]["y"], G.nodes[u]["x"]),
        (G.nodes[v]["y"], G.nodes[v]["x"])
    ]

    base_weight = 1.0 + 4.0 * (freq / max_freq)

    # 等比加粗 1 倍
    weight = base_weight * 2

    folium.PolyLine(
        locations=coords,
        color="white",
        weight=weight,
        opacity=1   # 不透明
    ).add_to(m)

# 点图层
fg_attraction = FeatureGroup(name="Attraction", show=True)
fg_restaurant = FeatureGroup(name="Restaurant", show=True)
fg_hotel = FeatureGroup(name="Hotel", show=True)
fg_shop = FeatureGroup(name="Shop", show=True)

layer_map = {
    "attraction": fg_attraction,
    "restaurant": fg_restaurant,
    "hotel": fg_hotel,
    "shop": fg_shop
}

# 只保留点，不加文字标注
for _, row in selected_df.iterrows():
    cat = row["category"]
    name = row["name"]
    lat = row["lat"]
    lon = row["lon"]
    media = row["media"]
    count_val = row["count"]

    color = CATEGORY_COLORS.get(cat, "#cccccc")
    radius = 7 if cat == "hotel" else 5

    popup_html = f"""
    <b>{name}</b><br>
    Category: {cat}<br>
    Source media: {media}<br>
    Count: {count_val}<br>
    Lat: {lat:.6f}<br>
    Lon: {lon:.6f}
    """

    folium.CircleMarker(
        location=(lat, lon),
        radius=radius,
        color=color,
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.95,
        popup=popup_html
    ).add_to(layer_map[cat])

# 不再添加 folium.Marker + DivIcon 文字标注

fg_attraction.add_to(m)
fg_restaurant.add_to(m)
fg_hotel.add_to(m)
fg_shop.add_to(m)

LayerControl(collapsed=False).add_to(m)

m.fit_bounds([[south, west], [north, east]])

legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background-color: rgba(0,0,0,0.78);
    color: white;
    padding: 12px 14px;
    border-radius: 8px;
    font-size: 14px;
    line-height: 1.6;
    box-shadow: 0 0 8px rgba(255,255,255,0.15);
">
    <div style="font-weight: bold; margin-bottom: 6px;">Generation Z Route Network</div>
    <div><span style="color:#6ec1ff;">●</span> Attraction</div>
    <div><span style="color:#ff6b6b;">●</span> Restaurant</div>
    <div><span style="color:#ffffff;">●</span> Hotel</div>
    <div><span style="color:#ffd166;">●</span> Shop</div>
    <div><span style="color:#ffffff;">━━</span> All-pairs walking shortest-path network</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print("\n地图已生成，下面直接在线显示。")
display(m)

# =========================================================
# 14. 地图后输出点位信息表
#    名称 / 来源库 / 类型
# =========================================================
point_info_table = (
    selected_df[["name", "media", "category"]]
    .rename(columns={
        "name": "点名称",
        "media": "来源库",
        "category": "类型"
    })
    .sort_values(["来源库", "类型", "点名称"])
    .reset_index(drop=True)
)

print("\n点位信息表：")
display(point_info_table)

本次动态随机种子: 4201707131

统一后的点位数据预览：


,media,category,name,count,lat,lon,source_file,sample_weight
0,paper manual magazine,attraction,Campanile di San Marco,6.0,45.434247,12.339075,Paper manual magazine_deduplicated (1).xlsx,6.0
1,paper manual magazine,attraction,Chiesa di San Moise,1.0,45.433594,12.336028,Paper manual magazine_deduplicated (1).xlsx,1.0
2,paper manual magazine,attraction,Galleria Franchetti,6.0,45.441354,12.334007,Paper manual magazine_deduplicated (1).xlsx,6.0
3,paper manual magazine,attraction,Galleria d'Arte Moderna,1.0,45.434135,12.333572,Paper manual magazine_deduplicated (1).xlsx,1.0
4,paper manual magazine,attraction,Gallerie Dell'Accademia,6.0,45.431414,12.328302,Paper manual magazine_deduplicated (1).xlsx,6.0



各媒介 / 各类别数量：


category,attraction,hotel,restaurant,shop
media,,,,
paper manual magazine,32,15,19,13
short video social media,37,21,33,22
tv official news,36,17,15,8
website blogs,56,30,32,33



本次选中的点：


...


按来源库透视统计：


category,attraction,hotel,restaurant,shop
media,,,,
paper manual magazine,3,1,1,2
tv official news,4,0,0,1
website blogs,1,0,1,0



正在下载城市步行路网...
路网下载完成。
节点数: 4755
边数: 12126

点对组合数量: 91
成功求得最短路径段数: 91
被使用的唯一街道路段数: 597

地图已生成，下面直接在线显示。



点位信息表：


,点名称,来源库,类型
0,Grand Canal,paper manual magazine,attraction
1,Museo del Settecento,paper manual magazine,attraction
2,Palazzo Ducale,paper manual magazine,attraction
3,ai nuovo teson,paper manual magazine,hotel
4,"Le Bistrot de Venise, Venetian, French",paper manual magazine,restaurant
5,Captain Candy,paper manual magazine,shop
6,Rialto Market,paper manual magazine,shop
7,Ca' Rezzonico,tv official news,attraction
8,Palazzo Mocenigo,tv official news,attraction
9,Rialto Bridge,tv official news,attraction
